# Линейная регрессия на реальных данных (продолжение)

Продолжаем разбирать регрессию на наблюдениях метеостанции Малые Кармакулы (начало — в `Lect03_LR_real_data.ipynb`). Здесь добавляются:

- **порождение признаков**: косинусы годового и суточного циклов вместо «сырых» номера дня и часа;
- **пайплайн** `Pipeline` — стандартизация и модель в одном объекте;
- **нелинейная модель** `CatBoostRegressor` (градиентный бустинг) в сравнении с `LinearRegression`;
- **разделение на тренировочную и тестовую выборки** и оценка качества на невиданных данных;
- **анализ коэффициентов** модели (только для линейной регрессии).

### Полезные ссылки

- [Пайплайны в sklearn (документация)](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html)
- [Пайплайны в sklearn (youtube)](https://www.youtube.com/watch?v=jzKSAeJpC6s)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [CatBoost: документация](https://catboost.ai/docs/) и [CatBoostRegressor](https://catboost.ai/docs/concepts/python-reference_catboostregressor.html)
- [Метрики регрессии (RMSE, R²) в sklearn](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)

## Импорт библиотек

Подключаем числовые и табличные инструменты, графику, а также пайплайн, масштабирование, разделение выборок, метрики и модели — линейную регрессию и градиентный бустинг CatBoost.

**Инструменты:** [NumPy](https://numpy.org/doc/stable/), [pandas](https://pandas.pydata.org/docs/), [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html), [sklearn.pipeline.Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics), [CatBoostRegressor](https://catboost.ai/docs/concepts/python-reference_catboostregressor.html).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

## Подготовка данных

И снова здравствуйте, Малые Кармакулы

Читаем таблицу, подготовленную на втором занятии (`Lect02_EDA_demo.ipynb`), и повторяем минимальную чистку: убираем `tg` (температура поверхности почвы — почти вся отсутствует), `td` и `psl` (дублируют другие признаки), затем удаляем строки с пропусками и сравниваем размеры таблицы до и после этого.

**Инструменты:** [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [DataFrame.drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html), [DataFrame.dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html), [DataFrame.shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html), [DataFrame.head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html).

In [ ]:
df = pd.read_csv('EDA_demo_data/karmakuly_processed.csv', parse_dates=['Date'], index_col='Date')
df.head()

#df = df['2015':]

print ('df shape with nans: ', df.shape)     # размер таблицы вместе с пропусками
# tg почти пуст, td и psl дублируют другие признаки
df = df.drop (['tg', 'td', 'psl'], axis=1) #, 'td', 
df = df.dropna()                             # удаляем строки с пропусками
print ('df shape without nans: ', df.shape)  # размер после чистки
df.head()

## Порождение дополнительных признаков

Добавляем два «циклических» признака вместо «сырых» номера дня года и часа:

- `year_pos` — косинус фазы годового цикла: $\cos(\pi + 2\pi \cdot \text{dayofyear} / 365)$, то есть −1 в начале января и +1 в середине лета;
- `day_pos` — косинус фазы суточного цикла: $\cos(\pi + 2\pi \cdot \text{hour} / 24)$, то есть −1 в полночь и +1 в полдень.

Такое представление гладкое: у «сырых» `dayofyear` и `hour` есть разрыв на границе года и суток, а у косинуса — нет, поэтому модели проще выучить плавный сезонный и суточный ход. Строим два графика: годовой ход за 2015 год и суточный ход за первые три дня 2015 года.

**Инструменты:** [DatetimeIndex.dayofyear](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.dayofyear.html), [DatetimeIndex.hour](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.hour.html), [numpy.cos](https://numpy.org/doc/stable/reference/generated/numpy.cos.html), [DataFrame.plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html), [matplotlib.pyplot.subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html).

In [ ]:
df['year_pos'] = np.cos (np.pi + 2 * np.pi * df.index.dayofyear/365)
df['day_pos']  = np.cos (np.pi + 2 * np.pi * df.index.hour/24)

fig, ax = plt.subplots (1,2, figsize=(10,5))

df['2015':]['year_pos'].plot(ax = ax[0], title='year_pos (annual cycle)',
                             xlabel='Дата', ylabel='year_pos (косинус фазы года)')
df['2015-01-01':'2015-01-03']['day_pos'].plot(ax = ax[1], title='day_pos (diurnal cycle)',
                                              xlabel='Дата', ylabel='day_pos (косинус фазы суток)')

## Обучение и проверка модели

Для начала посмотрим на оценки, если обучать и проверять модель на одних и тех же данных

Функция `evaluate_model` считает прогноз и рисует диаграмму «наблюдённое — предсказанное»: плотность точек показана шестиугольными корзинами, цвет — число наблюдений, а линия $y = x$ соответствует идеальному прогнозу. В заголовке выводятся RMSE и $R^2$.

Модель собирается в `Pipeline`: сначала `StandardScaler` (стандартизация признаков), затем регрессор. Сейчас в пайплайне стоит `CatBoostRegressor` — модель градиентного бустинга, которая умеет описывать нелинейные зависимости; закомментированная строка позволяет вернуться к `LinearRegression`.

Оценка на тех же данных, на которых модель обучалась, всегда завышена: модель уже «видела» правильные ответы. Поэтому в следующей ячейке данные делятся на тренировочную и тестовую выборки.

**Инструменты:** [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [CatBoostRegressor](https://catboost.ai/docs/concepts/python-reference_catboostregressor.html), [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [matplotlib.pyplot.hexbin](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hexbin.html), [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html), [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html).

In [ ]:
def evaluate_model (model, X, y):
       y_pred = model.predict(X)

       plt.figure()

       plt.hexbin (y, y_pred, mincnt=1, gridsize=50)

       plt.axis('equal')
       ax = plt.gca()
       lims = [
              np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
              np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
              ]
       plt.plot(lims, lims, 'k-', alpha=0.75, zorder=0)  # линия y = x — идеальный прогноз
       plt.xlim(lims)
       plt.ylim(lims)
       plt.grid()
       plt.xlabel('Наблюдённая температура воздуха, °C')
       plt.ylabel('Предсказанная температура воздуха, °C')
       plt.colorbar(label='Число наблюдений')

       r = np.corrcoef(y_pred, y)[0,1]
       r2 = r2_score(y, y_pred)
       mse = mean_squared_error(y, y_pred)

       plt.title ('RMSE = %.2f, R2 = %.2f'%(np.sqrt (mse), r2))

target_var = 'ta'

y = df[target_var]
X = df.drop(target_var, axis=1)

model = Pipeline([   # пайплайн: стандартизация признаков + модель
                ('scaler', StandardScaler()),
                #('regressor', LinearRegression())
                ('regressor', CatBoostRegressor(silent=True)) 
                ])

model.fit(X, y)
evaluate_model (model, X, y)

## Разделение на тестовую и тренировочную выборки

Делим данные на тренировочную (75 %) и тестовую (25 %) части. Модель обучается только на первой, а качество измеряется только на второй — так оценка показывает, как модель поведёт себя на данных, которых она не видела. Параметр `random_state` фиксирует разбиение, чтобы результат воспроизводился; при `shuffle=False` (закомментированный вариант) выборки делятся подряд по времени, без перемешивания — для временных рядов это иногда важнее.

**Инструменты:** [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [Pipeline.fit](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline.fit).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,  shuffle=False) 

#display(X_train)
#display(X_test)

model.fit(X_train, y_train)              # обучаем только на тренировочной выборке
evaluate_model (model, X_test, y_test)   # проверяем на тестовой выборке

## Анализ коэффициентов модели (для линейной регрессии)

Коэффициенты есть только у линейных моделей: у `LinearRegression` это атрибут `coef_` — вклад каждого признака в прогноз (на сколько изменится прогноз при увеличении признака на единицу). У `CatBoostRegressor` такого атрибута нет, поэтому в текущей конфигурации (CatBoost в пайплайне) ячейка ниже завершится ошибкой `AttributeError`; чтобы она сработала, верните в пайплайне `LinearRegression`. Доступ к шагу пайплайна — по имени: `model['regressor']`.

**Инструменты:** [LinearRegression.coef_](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.coef_), [доступ к шагам пайплайна](https://scikit-learn.org/stable/modules/compose.html#accessing-steps), [DataFrame](https://pandas.pydata.org/docs/reference/frame.html).

In [ ]:
# коэффициенты есть только у линейных моделей (у CatBoost атрибута coef_ нет)
coefs = pd.DataFrame(model['regressor'].coef_, X.columns, columns=['coef']) 
display(coefs)